# M02-03 — Calidad y limpieza

[← Anterior](03-lab-schema-tipos.ipynb) · [Siguiente →](../M03-transformacion-datos/01-teoria.ipynb)

Este fichero es el **guion**. No lo rellenes aquí: **crea tu propio notebook** y ve construyéndolo celda a celda.

## Qué vas a hacer

Aplicar las reglas intra-tabla y escribir el staging en Parquet. Si perdiste el notebook anterior, rehaz la lectura tipada de M02-02 al empezar.

## 0 — Crea tu notebook

1. En el explorador, abre la carpeta `notebooks/trabajo/`.
2. Clic derecho → **New File…**
3. Nombre exacto: `M02-03-calidad-limpieza.ipynb` (incluye `.ipynb`).
4. Ábrelo. Arriba a la derecha (o `F1` → `Notebook: Select Notebook Kernel`) elige **Python (NovaShop)**.
5. Deja **este** guion a un lado (pestaña) y escribe **solo** en el tuyo.

## Cómo organizar *tu* notebook (siempre)

En cada paso creas **dos celdas**, en este orden:

1. **Markdown** — qué vas a hacer y por qué, con tus palabras.
2. **Código** — el de la celda de código del paso. Lo ejecutas (`Shift+Enter`), miras la salida y, si no cuadra, lo mejoras.

No dejes un muro de código sin explicación. Un notebook se lee de arriba abajo, como un cuaderno.

> Kernel **Python (NovaShop)**. Si no aparece: terminal → `bash .devcontainer/setup.sh` → vuelve a elegir kernel.


## Contrato (cúmplelo tal cual)

| Tabla | Regla |
|-------|--------|
| `customers` | `country` vacío o nulo → `"UNK"`. **No** borres filas. |
| `products` | Tira filas con `list_price` nulo. |
| `orders` | Tira `customer_id` nulo o `""`. **No** tiras huérfanos `CX*`. |
| `order_items` | Tira `product_id` vacío o `qty <= 0`. |
| `events` | Tira `customer_id` nulo. |


### Paso 1 — Arranque y lecturas tipadas

En *tu* notebook: una celda Markdown que explique esto (con tus palabras):

Repito la lectura de M02-02 (orders con coalesce, items casteados, products, customers, events). No invento otro schema.

Debajo, una celda de código. El código está **en la celda siguiente** (márcalo y llévatelo). Ejecuta (`Shift+Enter`).

**Comprueba.** 800 250 60 2046 2500 (aún sucios).

**Por qué este paso.** El lab de limpieza parte de tipos ya puestos. Si saltas esto, los filtros no coinciden.


In [ ]:
import sys
from pathlib import Path

# El notebook puede estar en trabajo/; subimos hasta encontrar el repo.
_here = Path.cwd().resolve()
ROOT = next(
    p
    for p in [_here, *_here.parents]
    if (p / "labs" / "_shared" / "session.py").is_file()
)
sys.path.insert(0, str(ROOT / "labs" / "_shared"))

from paths import RAW, STAGING, CURATED  # rutas absolutas, no Path("data/raw")
from session import get_spark

print("ROOT   ", ROOT)
print("RAW    ", RAW, "existe:", RAW.is_dir())
print("STAGING", STAGING)
print("CURATED", CURATED)


from pyspark.sql.functions import col, coalesce, to_timestamp, when, trim
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DecimalType, TimestampType

spark = get_spark("novashop-m02")

orders = (
    spark.read.option("header", True).csv(str(RAW / "orders.csv"))
    .withColumnRenamed("OrderId", "order_id")
    .withColumnRenamed("CustomerId", "customer_id")
    .withColumnRenamed("OrderDate", "order_ts_raw")
    .withColumnRenamed("Status", "status")
    .withColumnRenamed("Channel", "channel")
    .withColumn(
        "order_ts",
        coalesce(
            to_timestamp(col("order_ts_raw"), "yyyy-MM-dd HH:mm:ss"),
            to_timestamp(col("order_ts_raw"), "dd/MM/yyyy"),
        ),
    )
    .drop("order_ts_raw")
)
customers = spark.read.option("header", True).csv(str(RAW / "customers.csv"))
products = (
    spark.read.option("multiLine", True).json(str(RAW / "products.json"))
    .withColumnRenamed("productId", "product_id")
    .withColumnRenamed("listPrice", "list_price")
    .withColumn("list_price", col("list_price").cast(DecimalType(10, 2)))
)
items = (
    spark.read.option("header", True).csv(str(RAW / "order_items.csv"))
    .withColumn("qty", col("qty").cast(IntegerType()))
    .withColumn("unit_price", col("unit_price").cast(DecimalType(10, 2)))
    .withColumn("discount", col("discount").cast(DecimalType(5, 2)))
)
events = spark.read.schema(
    StructType([
        StructField("event_id", StringType(), False),
        StructField("customer_id", StringType(), True),
        StructField("event_type", StringType(), True),
        StructField("ts", TimestampType(), True),
        StructField("session_id", StringType(), True),
        StructField("page", StringType(), True),
        StructField("product_id", StringType(), True),
    ])
).json(str(RAW / "events.jsonl"))
print(orders.count(), customers.count(), products.count(), items.count(), events.count())


### Paso 2 — Clientes y productos

En *tu* notebook: una celda Markdown que explique esto (con tus palabras):

País vacío es recuperable (UNK). Producto sin precio no se vende: se tira.

Debajo, una celda de código. El código está **en la celda siguiente** (márcalo y llévatelo). Ejecuta (`Shift+Enter`).

**Comprueba.** customers **250** (5 `UNK`) · products **57**.

**Por qué este paso.** Spark CSV convierte vacíos en null: hay que tratar null y `""`.


In [ ]:
customers_clean = customers.withColumn(
    "country",
    when(col("country").isNull() | (trim(col("country")) == ""), "UNK").otherwise(col("country")),
)
products_clean = products.where(col("list_price").isNotNull())
print("customers", customers_clean.count(), "unk", customers_clean.where(col("country") == "UNK").count())
print("products", products_clean.count())


### Paso 3 — Pedidos, líneas y eventos

En *tu* notebook: una celda Markdown que explique esto (con tus palabras):

Claves vacías rompen joins. Los huérfanos CX* y P999 se quedan: M04 los visibiliza.

Debajo, una celda de código. El código está **en la celda siguiente** (márcalo y llévatelo). Ejecuta (`Shift+Enter`).

**Comprueba.** orders **788** · items **2010** · events **2420**.

**Por qué este paso.** Si filtras también los CX* no te saldrá 788.


In [ ]:
orders_clean = orders.where(trim(col("customer_id")) != "")
items_clean = items.where((trim(col("product_id")) != "") & (col("qty") > 0))
events_clean = events.where(col("customer_id").isNotNull())
print("orders", orders_clean.count())
print("items", items_clean.count())
print("events", events_clean.count())


### Paso 4 — Escribe staging (Parquet)

En *tu* notebook: una celda Markdown que explique esto (con tus palabras):

Parquet guarda el schema. El siguiente módulo no vuelve a inferir CSV.

Debajo, una celda de código. El código está **en la celda siguiente** (márcalo y llévatelo). Ejecuta (`Shift+Enter`).

**Comprueba.** Cinco carpetas bajo `data/staging/`. Releer `orders_clean` = 788 y `order_ts` timestamp.

**Por qué este paso.** Si escribes CSV “para verlo”, pierdes tipos.


In [ ]:
STAGING.mkdir(parents=True, exist_ok=True)
pairs = {
    "customers_clean": customers_clean,
    "products_clean": products_clean,
    "orders_clean": orders_clean,
    "order_items_clean": items_clean,
    "events_clean": events_clean,
}
for name, frame in pairs.items():
    dest = STAGING / name
    frame.write.mode("overwrite").parquet(str(dest))
    print(name, dest)
print("releer orders", spark.read.parquet(str(STAGING / "orders_clean")).count())


## Comprueba

Antes de dar el lab por cerrado, vuelve a ejecutar de arriba abajo (**Run All**) y verifica:

En `orders_clean` e `items_clean`, nulos de `order_id` / `customer_id` / `product_id` → **0**.
Pedidos **788**. Líneas **2010**.


## Mejora — Filas que tiraste

Imprime `antes - después` de cada regla (pedidos, líneas, eventos, productos). Markdown que interprete cada diferencia.

Si te atasca, el código está en la celda siguiente.


In [ ]:
orders     800 - 788 = 12   (customer_id vacío)
items     2046 - 2010 = 36  (21 sin producto ∪ 15 qty 0)
events    2500 - 2420 = 80
products    60 -  57 = 3


## Si algo falla

| Qué ves | Suele ser | Qué haces |
|---------|-----------|-----------|
| 788 no sale | Filtraste también los CX* | Esos 8 se quedan; solo quitas customer_id vacío |
| items ≠ 2010 | Filtros a medias | Una sola where: producto no vacío **y** qty > 0 |
| Staging ilegible | Escribiste CSV | Parquet; para espiar: `spark.read.parquet(...).show()` |


## Siguiente

Cuando hayas **comprobado** y (si quieres) **mejorado**, abre [M03 — teoría](../M03-transformacion-datos/01-teoria.ipynb).
